# Pakistani Social Media Content Detective 🇵🇰

**SYNTHETIC DEMONSTRATION ONLY.** No empirical claims about Pakistani audiences are established here.

Business question: What type of social-media content gets Pakistani audiences to engage?

This notebook runs the same import, cleaning, EDA, caption sentiment, and recommendation code used by the dashboard. Run all cells from this directory or the repository root. Replace the input path **and manifest** only after inspecting a genuine dataset and its provenance.

In [1]:
from pathlib import Path
import sys
import json
import pandas as pd
import plotly.express as px
from IPython.display import display, Markdown

ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
from src.loader import Manifest, read_csv
from src.schema import suggest_mapping, apply_mapping
from src.analysis import prepare, filter_posts, feasibility, dataset_label
from src.metrics import choose_metric, metric_coverage, summarize
from src.hashtags import hashtag_table
from src.reporting import summary_tables, executive_summary, markdown_report
from src.recommendations import recommend

manifest = Manifest(**json.loads((ROOT / "data/demo_manifest.json").read_text()))
raw = read_csv((ROOT / "data/sample_synthetic_posts.csv").read_bytes())
mapping = suggest_mapping(raw.columns)
cleaned, quality = prepare(apply_mapping(raw, mapping), manifest)
posts = filter_posts(cleaned)
metric = choose_metric(posts).name
print(dataset_label(posts), "|", len(posts), "posts | primary metric:", metric)
display(pd.DataFrame([manifest.to_dict()]))

SYNTHETIC DEMO | 180 posts | primary metric: rate_impressions


,file_name,source_name,source_type,status,collected_on,geographic_relevance,license_notes,source_timezone,audience_evidence,notes,sha256
0,sample_synthetic_posts.csv,Fixed-seed demo generator,synthetic,synthetic,,Fictional Pakistan-market scenario; no verifie...,Generated project fixture; not scraped from pl...,Asia/Karachi,,Seed 42. All 180 observations and accounts are...,


## Validate before interpreting

Identical original duplicates are reported and removed. Conflicting source/account/post IDs stay in cleaned exports and are excluded from analysis. Missing and invalid counts remain missing. Rates require complete likes/comments/shares and a positive, separately defined denominator.

In [2]:
display(pd.DataFrame({"field": list(quality["missingness"]), "missing": list(quality["missingness"].values())}))
print({k:v for k,v in quality.items() if k != "missingness"})
display(metric_coverage(posts))
display(feasibility(posts, metric, manifest))

,field,missing
0,post_id,0
1,source_url,180
2,source_name,0
3,source_type,0
4,is_synthetic,0
5,brand,0
6,platform,0
7,post_datetime,0
8,timezone_source,0
9,content_type,0


{'input_rows': 180, 'cleaned_rows': 180, 'exact_duplicates_removed': 0, 'conflicting_rows_excluded': 0, 'generated_local_ids': 0, 'invalid_counts': {'likes': 0, 'comments': 0, 'shares': 0, 'views': 0, 'impressions': 0, 'reach': 0, 'follower_count': 0}, 'invalid_timestamps': 0, 'timezone_unknown': 0, 'original_columns': ['post_id', 'source_url', 'source_name', 'source_type', 'is_synthetic', 'brand', 'platform', 'post_datetime', 'timezone_source', 'content_type', 'caption', 'hashtags', 'topic', 'caption_language', 'creative_angle', 'local_context', 'likes', 'comments', 'shares', 'views', 'impressions', 'reach', 'follower_count', 'audience_location', 'audience_segment', 'caption_sentiment', 'collection_datetime']}


,metric,definition,valid_posts,total_posts
0,available_interactions,Sum of observed likes/comments/shares per post...,180,180
1,complete_interactions,"likes + comments + shares, only when all three...",154,180
2,visible_interactions,"Visible interactions (likes + comments), only ...",180,180
3,visible_per_1000_views,"1,000 × (likes + comments) / views; both count...",180,180
4,rate_impressions,"100 × complete interactions / impressions, wit...",154,180
5,rate_reach,"100 × complete interactions / reach, with posi...",154,180
6,rate_views,Per-view interaction rate: 100 × complete inte...,154,180
7,rate_followers,100 × complete interactions / follower count; ...,0,180
8,likes,Observed likes only,180,180
9,comments,Observed comments only,180,180


,question,support,explanation
0,Which platform performs better?,Supported,Requires at least two platforms with valid mea...
1,Which content type gets more engagement?,Supported,Content type and valid measurements required; ...
2,What topics and hashtags are popular?,Partial,Observed caption/tag/topic usage only; frequen...
3,What sentiment is associated with engagement?,Partial,Caption sentiment only. Supplied labels/baseli...
4,What posting time/day appears better?,Supported,Known timezone required for Karachi weekday/ho...
5,Which Pakistani audience segment seems more re...,Unavailable,"Requires documented segment evidence, declared..."
6,What should a Pakistani brand post next?,Partial,Descriptive experiments only; no causal or pre...


## 1–2. Platforms and content formats

Compare means, medians, valid n, coverage, and group date ranges. Raw platform counts and differently defined denominators require qualification. Filter to comparable accounts/date ranges before forming a practical experiment.

In [3]:
tables = summary_tables(posts, metric, manifest)
for group in ["platform", "content_type"]:
    display(tables[group])
    px.box(posts, x=group, y=metric, points="outliers", title=f"SYNTHETIC DEMO: {group} vs {metric}").show()

,platform,total_posts,n,coverage_pct,mean,median,small_sample,first_date,last_date
0,TikTok,60,52,86.666667,5.763181,4.213220,False,2026-08-01,2026-09-29
1,Instagram,60,51,85.000000,6.472972,3.661360,False,2026-08-01,2026-09-29
2,Facebook,60,51,85.000000,5.348945,3.210383,False,2026-08-01,2026-09-29


,content_type,total_posts,n,coverage_pct,mean,median,small_sample,first_date,last_date
0,reel,61,54,88.524590,7.132749,5.039872,False,2026-08-01,2026-09-29
1,carousel,67,56,83.582090,4.470191,3.727418,False,2026-08-01,2026-09-29
2,image,52,44,84.615385,6.070546,3.600432,False,2026-08-01,2026-09-27


## 3. Hashtag usage and supplied topics

Unicode-safe extraction counts each tag once per post. Frequency describes usage in this sample, not audience popularity. No topics are invented.

In [4]:
display(hashtag_table(posts, metric))
display(tables["topic"])

,hashtag,total_posts,n,coverage_pct,mean,median,small_sample
0,#communitydemo,31,25,80.645161,7.327635,5.269171,False
6,#پاکستان,61,49,80.327869,7.229595,4.878049,False
3,#saledemo,30,26,86.666667,6.666378,4.759890,False
4,#servicedemo,27,23,85.185185,5.517491,4.235294,False
5,#styledemo,95,83,87.368421,5.685932,3.787451,False
1,#craftdemo,27,23,85.185185,5.242506,3.304366,False
2,#guidedemo,35,33,94.285714,3.865069,3.123118,False


,topic,total_posts,n,coverage_pct,mean,median,small_sample
1,community,31,25,80.645161,7.327635,5.269171,False
4,promotion,30,26,86.666667,6.666378,4.759890,False
5,service update,27,23,85.185185,5.517491,4.235294,False
2,new collection,30,24,80.000000,7.127469,3.938684,False
0,behind the scenes,27,23,85.185185,5.242506,3.304366,False
3,product guide,35,33,94.285714,3.865069,3.123118,False


## 4. Caption sentiment

Supplied labels are disclosed as unverified. Otherwise a tiny offline English lexicon abstains on Urdu script, uncertain vocabulary, and negation. Roman Urdu accuracy is not claimed. These labels describe captions, not audience comments.

In [5]:
display(posts.sentiment_label.value_counts().rename_axis("caption sentiment").to_frame("posts"))
display(tables["sentiment_label"])
display(posts[["caption_original", "sentiment_label", "sentiment_method"]].head(12))

,posts
caption sentiment,
unassessed,88
positive,35
neutral,30
negative,27


,sentiment_label,total_posts,n,coverage_pct,mean,median,small_sample
1,neutral,30,26,86.666667,6.666378,4.759890,False
0,negative,27,23,85.185185,5.517491,4.235294,False
2,positive,35,33,94.285714,3.865069,3.123118,False


,caption_original,sentiment_label,sentiment_method
0,Discover our beautiful new collection #StyleDe...,unassessed,Urdu/Arabic-script text: unsupported by Englis...
1,Discover our beautiful new collection #StyleDe...,unassessed,Urdu/Arabic-script text: unsupported by Englis...
2,We are disappointed with the poor service #Ser...,negative,Tiny English lexicon baseline (unvalidated)
3,Aaj hamari nayi collection dekhein #CraftDemo,unassessed,Language/vocabulary uncertain; Roman Urdu is n...
4,We are disappointed with the poor service #Ser...,negative,Tiny English lexicon baseline (unvalidated)
5,We are disappointed with the poor service #Ser...,negative,Tiny English lexicon baseline (unvalidated)
6,Discover our beautiful new collection #StyleDe...,unassessed,Urdu/Arabic-script text: unsupported by Englis...
7,آپ کی پسند کیا ہے؟ #پاکستان #CommunityDemo,unassessed,Urdu/Arabic-script text: unsupported by Englis...
8,Aaj hamari nayi collection dekhein #CraftDemo,unassessed,Language/vocabulary uncertain; Roman Urdu is n...
9,Aaj hamari nayi collection dekhein #CraftDemo,unassessed,Language/vocabulary uncertain; Roman Urdu is n...


## 5–6. Timing and audience evidence

Karachi-local timing requires a known source timezone/offset. Date-only observations do not imply an hour. The synthetic fixture has no audience demographics. Pakistani brands alone would still only be a market proxy.

In [6]:
display(posts.time_status.value_counts().to_frame("posts"))
display(tables["weekday"])
display(tables["hour"])
display(tables["audience_segment"])
print("Insufficient audience-demographic data" if tables["audience_segment"].empty else "See audience evidence and grain limitations")

,posts
time_status,
known,180


,weekday,total_posts,n,coverage_pct,mean,median,small_sample
1,Monday,27,18,66.666667,6.848033,5.292498,False
3,Sunday,27,27,100.000000,6.319625,4.483683,False
4,Thursday,24,24,100.000000,6.979227,3.977883,False
5,Tuesday,27,27,100.000000,5.044328,3.953432,False
0,Friday,24,24,100.000000,5.149578,3.916209,False
6,Wednesday,24,16,66.666667,6.345983,3.724405,False
2,Saturday,27,18,66.666667,4.438043,3.042681,False


,hour,total_posts,n,coverage_pct,mean,median,small_sample
0,9,48,46,95.833333,6.446109,4.359489,False
1,13,63,53,84.126984,5.772989,4.036196,False
2,19,69,55,79.710145,5.456613,3.393686,False


,audience_segment,total_posts,n,coverage_pct,mean,median,small_sample


Insufficient audience-demographic data


## 7. What should a brand test next?

The deterministic engine quotes computed medians, means, comparison n, metric, dates, dataset status, and limitations. Groups need n≥5 and ≥80% metric coverage; otherwise it asks for observations. Association is not causation.

In [7]:
for card in recommend(posts, metric, manifest):
    display(Markdown(f"### {card['kind']}\n{card['action']}\n\nEvidence: {card['evidence']}\n\nConfidence: {card['confidence']}. {card['limitations']}"))

### Format experiment
Test reel against carousel, holding brand, platform, campaign spend, and observation window constant.

Evidence: SYNTHETIC DEMO. 2026-08-01 to 2026-09-29 (source calendar dates). Active posts: 180. rate_impressions: reel median 5.04, mean 7.13, n=54/61; carousel median 3.73, mean 4.47, n=56/67. Definition: 100 × complete interactions / impressions, with positive impressions.

Confidence: Exploratory; requires validation. Observational association; account size, paid boosts, post age, and selection may confound results. No proven outcomes or causal effect. Synthetic demonstration only; no empirical marketing conclusion. Groups may cover different accounts, platforms, or dates; narrow filters before testing.

### Candidate hashtags
Test an otherwise similar post using #communitydemo versus #پاکستان; hashtag association may reflect the underlying topic.

Evidence: SYNTHETIC DEMO. 2026-08-01 to 2026-09-29 (source calendar dates). Active posts: 180. rate_impressions: #communitydemo median 5.27, mean 7.33, n=25/31; #پاکستان median 4.88, mean 7.23, n=49/61. Definition: 100 × complete interactions / impressions, with positive impressions.

Confidence: Exploratory; requires validation. Observational association; account size, paid boosts, post age, and selection may confound results. No proven outcomes or causal effect. Synthetic demonstration only; no empirical marketing conclusion. Groups may cover different accounts, platforms, or dates; narrow filters before testing.

### Candidate themes
Test a community theme against promotion using comparable creative and audience targeting.

Evidence: SYNTHETIC DEMO. 2026-08-01 to 2026-09-29 (source calendar dates). Active posts: 180. rate_impressions: community median 5.27, mean 7.33, n=25/31; promotion median 4.76, mean 6.67, n=26/30. Definition: 100 × complete interactions / impressions, with positive impressions.

Confidence: Exploratory; requires validation. Observational association; account size, paid boosts, post age, and selection may confound results. No proven outcomes or causal effect. Synthetic demonstration only; no empirical marketing conclusion. Groups may cover different accounts, platforms, or dates; narrow filters before testing.

### Caption language experiment
Test a Urdu caption against English, keeping message, platform, creative, and observation age comparable.

Evidence: SYNTHETIC DEMO. 2026-08-01 to 2026-09-29 (source calendar dates). Active posts: 180. rate_impressions: Urdu median 5.27, mean 7.33, n=25/31; English median 4.03, mean 5.65, n=106/122. Definition: 100 × complete interactions / impressions, with positive impressions.

Confidence: Exploratory; requires validation. Observational association; account size, paid boosts, post age, and selection may confound results. No proven outcomes or causal effect. Synthetic demonstration only; no empirical marketing conclusion. Groups may cover different accounts, platforms, or dates; narrow filters before testing.

### Local context experiment
Collect more comparable observations before choosing a winner.

Evidence: SYNTHETIC DEMO. 2026-08-01 to 2026-09-29 (source calendar dates). Active posts: 180. Need two groups with n ≥ 5 and ≥80% metric coverage.

Confidence: Insufficient evidence. Observational association; account size, paid boosts, post age, and selection may confound results. No proven outcomes or causal effect. Synthetic demonstration only; no empirical marketing conclusion. Insufficient comparison evidence.

### Creative approach experiment
Test Question / conversation against Offer / promotion for the same product, platform, and audience targeting.

Evidence: SYNTHETIC DEMO. 2026-08-01 to 2026-09-29 (source calendar dates). Active posts: 180. rate_impressions: Question / conversation median 5.27, mean 7.33, n=25/31; Offer / promotion median 4.76, mean 6.67, n=26/30. Definition: 100 × complete interactions / impressions, with positive impressions.

Confidence: Exploratory; requires validation. Observational association; account size, paid boosts, post age, and selection may confound results. No proven outcomes or causal effect. Synthetic demonstration only; no empirical marketing conclusion. Groups may cover different accounts, platforms, or dates; narrow filters before testing.

### Tentative posting window
Tentatively test Sunday 13:00 Asia/Karachi against Wednesday 9:00 Asia/Karachi; collect repeated observations in each slot.

Evidence: SYNTHETIC DEMO. 2026-08-01 to 2026-09-29 (source calendar dates). Active posts: 180. rate_impressions: Sunday 13:00 Asia/Karachi median 4.84, mean 5.00, n=11/11; Wednesday 9:00 Asia/Karachi median 4.57, mean 4.84, n=6/6. Definition: 100 × complete interactions / impressions, with positive impressions.

Confidence: Exploratory; requires validation. Observational association; account size, paid boosts, post age, and selection may confound results. No proven outcomes or causal effect. Synthetic demonstration only; no empirical marketing conclusion. Groups may cover different accounts, platforms, or dates; narrow filters before testing.

### Success metric
Pre-register rate_impressions and compare post-level medians in an A/B content experiment after an equal observation period.

Evidence: SYNTHETIC DEMO. 2026-08-01 to 2026-09-29 (source calendar dates). Active posts: 180. Valid observations: 154/180. 100 × complete interactions / impressions, with positive impressions.

Confidence: Experiment design only. Observational association; account size, paid boosts, post age, and selection may confound results. No proven outcomes or causal effect. Synthetic demonstration only; no empirical marketing conclusion. Keep sample size, denominator definition, account, and paid exposure comparable. A pilot is not a guaranteed lift.

## Reproduce the export

The executive summary covers all seven questions, including unsupported ones. The report below stays in memory; use the dashboard for explicit downloads. Demo findings must not be rewritten as empirical Pakistani conclusions.

In [8]:
report = markdown_report(posts, quality, metric, manifest)
assert "SYNTHETIC DEMO" in report
assert "Which Pakistani audience segment" in report
display(Markdown(executive_summary(posts, metric, manifest)))
print("Markdown report ready:", len(report), "characters")

### 1. Which platform performs better?
Supported: Highest eligible observed median: TikTok (4.21 rate_impressions; n=52/60). Requires at least two platforms with valid measurements; rates and raw counts have comparability limits.

### 2. Which content type gets more engagement?
Supported: Highest eligible observed median: reel (5.04 rate_impressions; n=54/61). Content type and valid measurements required; compare sample sizes and coverage.

### 3. What topics and hashtags are popular?
Partial: Most frequently used observed hashtag: #styledemo (95 posts). Usage is not audience popularity. Observed caption/tag/topic usage only; frequency is not audience popularity.

### 4. What sentiment is associated with engagement?
Partial: Highest eligible observed median: neutral (4.76 rate_impressions; n=26/30). Caption sentiment only. Supplied labels/baseline accuracy is unverified; no linked comments dataset.

### 5. What posting time/day appears better?
Supported: Highest eligible observed median: Sunday (4.48 rate_impressions; n=27/27). Known timezone required for Karachi weekday/hour analysis; date-only rows cannot support hours.

### 6. Which Pakistani audience segment seems more responsive?
Unavailable: Not answerable with the current data. Requires documented segment evidence, declared-real data, and explicit Pakistan audience location. Post-level context is not demographic response rate.

### 7. What should a Pakistani brand post next?
Propose a controlled content experiment from the evidence below; collect more data when comparisons are insufficient.

Markdown report ready: 17694 characters
